# **Climate Data - ETL**

This notebook transforms the 107 provincial climate source files into a single monthly province-level dataset.

The raw datasets are stored as gzip-compressed CSV objects in Google Cloud Storage. Each source object is read directly into pandas, processed through `transform_province_dataframe()`, assigned to the corresponding production-area and regional labels, and added to the final dataset.

Most output areas are based on one source file. Where historical administrative boundaries differ between the climate and production datasets, multiple source files are read and concatenated before the provincial transformation is applied.

## Environment and data locations

The notebook uses a hybrid data layout:

- the 107 raw climate datasets are stored in Google Cloud Storage;
- the notebook and the reusable transformation module are executed locally;
- the final transformed dataset is saved in the local project `data/` directory (to be later uploaded in a BigQuery table).

The source objects follow this structure:

```text
gs://agriclimate-intelligence-data/
└── raw/
    └── climate/
        └── v1/
            ├── agrigento.csv.gz
            ├── alessandria.csv.gz
            ├── ...
            └── <province>.csv.gz
```

The relevant local project structure is:

```text
project/
├── notebooks/
│   └── ETL_Climate_Data.ipynb
└── outputs/
    └── esportazione_full.csv
```

The local `outputs/` directory must already exist, and `province_transformation.py` must be importable from the active Python environment. Direct access to Google Cloud Storage also requires `gcsfs` and valid Google Cloud Application Default Credentials.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent

# Make the project package available to the notebook.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.province_transformation import transform_province_dataframe

In [2]:
# Resolve the local project root from the notebook location.
# The notebook is expected to be executed from the project's notebooks directory.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent

# Local directory used for the final transformed dataset
OUTPUT_DIR = PROJECT_ROOT / "data"

# Google Cloud resources containing the raw climate datasets
PROJECT_ID = "agriclimate-intelligence"
BUCKET_NAME = "agriclimate-intelligence-data"

# GCS prefix containing the 107 gzip-compressed provincial source files
INPUT_DIR = f"gs://{BUCKET_NAME}/raw/climate/v1"

# Options passed by pandas to gcsfs.
# Authentication is provided through Google Cloud Application Default Credentials.
GCS_STORAGE_OPTIONS = {
    "project": PROJECT_ID
}

## Production-area and source-file mapping

`PRODUCTION_AREA_ROWS` links each target production area to one or more provincial climate source files and to the province and region labels required in the final dataset.

The mapping retains the original `.csv` filenames because they are also passed to the transformation function as source identifiers. The additional `.gz` suffix used by the objects stored in Google Cloud Storage is added only when the full input path is constructed.

For production areas associated with multiple source files, the corresponding datasets are concatenated before applying the provincial transformation.


In [3]:
PRODUCTION_AREA_ROWS = [
    ("ITG14", ("agrigento.csv",), "Agrigento", "Sicilia"),
    ("ITC18", ("alessandria.csv",), "Alessandria", "Piemonte"),
    ("ITI32", ("ancona.csv",), "Ancona", "Marche"),
    ("ITI18", ("arezzo.csv",), "Arezzo", "Toscana"),
    ("ITI34", ("ascoli_piceno.csv",), "Ascoli Piceno", "Marche"),
    ("ITC17", ("asti.csv",), "Asti", "Piemonte"),
    ("ITF34", ("avellino.csv",), "Avellino", "Campania"),
    ("ITF47", ("bari.csv",), "Bari", "Puglia"),
    (
        "ITF48",
        ("barletta_andria_trani.csv",),
        "Barletta-Andria-Trani",
        "Puglia",
    ),
    ("ITH33", ("belluno.csv",), "Belluno", "Veneto"),
    ("ITF32", ("benevento.csv",), "Benevento", "Campania"),
    ("ITC46", ("bergamo.csv",), "Bergamo", "Lombardia"),
    ("ITC13", ("biella.csv",), "Biella", "Piemonte"),
    ("ITH55", ("bologna.csv",), "Bologna", "Emilia-Romagna"),
    ("ITC47", ("brescia.csv",), "Brescia", "Lombardia"),
    ("ITF44", ("brindisi.csv",), "Brindisi", "Puglia"),
    ("ITG15", ("caltanissetta.csv",), "Caltanissetta", "Sicilia"),
    ("ITF21", ("campobasso.csv",), "Campobasso", "Molise"),
    ("ITF31", ("caserta.csv",), "Caserta", "Campania"),
    ("ITG17", ("catania.csv",), "Catania", "Sicilia"),
    ("ITF63", ("catanzaro.csv",), "Catanzaro", "Calabria"),
    ("ITF14", ("chieti.csv",), "Chieti", "Abruzzo"),
    ("ITC42", ("como.csv",), "Como", "Lombardia"),
    ("ITF61", ("cosenza.csv",), "Cosenza", "Calabria"),
    ("ITC4A", ("cremona.csv",), "Cremona", "Lombardia"),
    ("ITF62", ("crotone.csv",), "Crotone", "Calabria"),
    ("ITC16", ("cuneo.csv",), "Cuneo", "Piemonte"),
    ("ITG16", ("enna.csv",), "Enna", "Sicilia"),
    ("ITI35", ("fermo.csv",), "Fermo", "Marche"),
    ("ITH56", ("ferrara.csv",), "Ferrara", "Emilia-Romagna"),
    ("ITI14", ("firenze.csv",), "Firenze", "Toscana"),
    ("ITF46", ("foggia.csv",), "Foggia", "Puglia"),
    (
        "ITH58",
        ("forli_cesena.csv",),
        "Forlì-Cesena",
        "Emilia-Romagna",
    ),
    ("ITI45", ("frosinone.csv",), "Frosinone", "Lazio"),
    ("ITC33", ("genova.csv",), "Genova", "Liguria"),
    (
        "ITH43",
        ("gorizia.csv",),
        "Gorizia",
        "Friuli-Venezia Giulia",
    ),
    ("ITI1A", ("grosseto.csv",), "Grosseto", "Toscana"),
    ("ITC31", ("imperia.csv",), "Imperia", "Liguria"),
    ("ITF22", ("isernia.csv",), "Isernia", "Molise"),
    ("ITC34", ("la spezia.csv",), "La Spezia", "Liguria"),
    ("ITF11", ("aquila.csv",), "L'Aquila", "Abruzzo"),
    ("ITI44", ("latina.csv",), "Latina", "Lazio"),
    ("ITF45", ("lecce.csv",), "Lecce", "Puglia"),
    ("ITC43", ("lecco.csv",), "Lecco", "Lombardia"),
    ("ITI16", ("livorno.csv",), "Livorno", "Toscana"),
    ("ITC49", ("lodi.csv",), "Lodi", "Lombardia"),
    ("ITI12", ("lucca.csv",), "Lucca", "Toscana"),
    ("ITI33", ("macerata.csv",), "Macerata", "Marche"),
    ("ITC4B", ("mantova.csv",), "Mantova", "Lombardia"),
    ("ITI11", ("massa_carrara.csv",), "Massa-Carrara", "Toscana"),
    ("ITF52", ("matera.csv",), "Matera", "Basilicata"),
    ("ITG13", ("messina.csv",), "Messina", "Sicilia"),
    ("ITC4C", ("milano.csv",), "Milano", "Lombardia"),
    ("ITH54", ("modena.csv",), "Modena", "Emilia-Romagna"),
    (
        "ITC4D",
        ("monza brianza.csv",),
        "Monza e della Brianza",
        "Lombardia",
    ),
    ("ITF33", ("napoli.csv",), "Napoli", "Campania"),
    ("ITC15", ("novara.csv",), "Novara", "Piemonte"),
    ("ITG28", ("oristano.csv",), "Oristano", "Sardegna"),
    ("ITG26", ("nuoro.csv",), "Nuoro", "Sardegna"),
    ("ITH36", ("padova.csv",), "Padova", "Veneto"),
    ("ITG12", ("palermo.csv",), "Palermo", "Sicilia"),
    ("ITH52", ("parma.csv",), "Parma", "Emilia-Romagna"),
    ("ITC48", ("pavia.csv",), "Pavia", "Lombardia"),
    ("ITI21", ("perugia.csv",), "Perugia", "Umbria"),
    (
        "ITI31",
        ("pesaro_urbino.csv",),
        "Pesaro e Urbino",
        "Marche",
    ),
    ("ITF13", ("pescara.csv",), "Pescara", "Abruzzo"),
    ("ITH51", ("piacenza.csv",), "Piacenza", "Emilia-Romagna"),
    ("ITI17", ("pisa.csv",), "Pisa", "Toscana"),
    ("ITI13", ("pistoia.csv",), "Pistoia", "Toscana"),
    (
        "ITH41",
        ("pordenone.csv",),
        "Pordenone",
        "Friuli-Venezia Giulia",
    ),
    ("ITF51", ("potenza.csv",), "Potenza", "Basilicata"),
    ("ITI15", ("prato.csv",), "Prato", "Toscana"),
    (
        "ITH10",
        ("bolzano.csv",),
        "Provincia Autonoma di Bolzano / Autonome Provinz Bozen",
        "Trentino-Alto Adige",
    ),
    (
        "SUD_SARDEGNA_AGG",
        ("cagliari.csv", "sud_sardegna.csv"),
        # This area combines Cagliari, Medio Campidano and
        # Carbonia-Iglesias because the weather and production
        # datasets use different historical administrative boundaries.
        "Area vasta Sud Sardegna",
        "Sardegna",
    ),
    ("ITG18", ("ragusa.csv",), "Ragusa", "Sicilia"),
    ("ITH57", ("ravenna.csv",), "Ravenna", "Emilia-Romagna"),
    (
        "ITF65",
        ("reggio_calabria.csv",),
        "Reggio Calabria",
        "Calabria",
    ),
    (
        "ITH53",
        ("reggio_emilia.csv",),
        "Reggio nell'Emilia",
        "Emilia-Romagna",
    ),
    ("ITI42", ("rieti.csv",), "Rieti", "Lazio"),
    ("ITH59", ("rimini.csv",), "Rimini", "Emilia-Romagna"),
    ("ITI43", ("roma.csv",), "Roma", "Lazio"),
    ("ITH37", ("rovigo.csv",), "Rovigo", "Veneto"),
    ("ITF35", ("salerno.csv",), "Salerno", "Campania"),
    ("ITG25", ("sassari.csv",), "Sassari", "Sardegna"),
    ("ITC32", ("savona.csv",), "Savona", "Liguria"),
    ("ITI19", ("siena.csv",), "Siena", "Toscana"),
    ("ITG19", ("siracusa.csv",), "Siracusa", "Sicilia"),
    ("ITC44", ("sondrio.csv",), "Sondrio", "Lombardia"),
    ("ITF43", ("taranto.csv",), "Taranto", "Puglia"),
    ("ITF12", ("teramo.csv",), "Teramo", "Abruzzo"),
    ("ITI22", ("terni.csv",), "Terni", "Umbria"),
    ("ITC11", ("torino.csv",), "Torino", "Piemonte"),
    ("ITG11", ("trapani.csv",), "Trapani", "Sicilia"),
    (
        "ITH20",
        ("trento.csv",),
        "Trento",
        "Trentino-Alto Adige",
    ),
    ("ITH34", ("treviso.csv",), "Treviso", "Veneto"),
    (
        "ITH44",
        ("trieste.csv",),
        "Trieste",
        "Friuli-Venezia Giulia",
    ),
    (
        "ITH42",
        ("udine.csv",),
        "Udine",
        "Friuli-Venezia Giulia",
    ),
    (
        "ITC20",
        ("aosta.csv",),
        "Valle d'Aosta / Vallée d'Aoste",
        "Valle d'Aosta",
    ),
    ("ITC41", ("varese.csv",), "Varese", "Lombardia"),
    ("ITH35", ("venezia.csv",), "Venezia", "Veneto"),
    (
        "ITC14",
        ("verbano_cusio_ossola.csv",),
        "Verbano-Cusio-Ossola",
        "Piemonte",
    ),
    ("ITC12", ("vercelli.csv",), "Vercelli", "Piemonte"),
    ("ITH31", ("verona.csv",), "Verona", "Veneto"),
    (
        "ITF64",
        ("vibo_valentino.csv",),
        "Vibo Valentia",
        "Calabria",
    ),
    ("ITH32", ("vicenza.csv",), "Vicenza", "Veneto"),
    ("ITI41", ("viterbo.csv",), "Viterbo", "Lazio"),
]

## Direct read from Google Cloud Storage

Each mapped filename is resolved to the corresponding `.csv.gz` object under the configured Google Cloud Storage prefix.

`pandas.read_csv()` opens the object through `gcsfs`, transfers the compressed data while the notebook runs and decompresses them during reading. No persistent local copy of the raw source file is created.

The remainder of the pipeline operates on the resulting pandas DataFrames in the same way as when the source files were read from the local filesystem.


In [4]:
# Collect the transformed monthly datasets before the final concatenation
monthly_datasets = []

In [5]:
for row in PRODUCTION_AREA_ROWS:
    source_frames = []

    for FILENAME in row[1]:
        # The mapping stores the original .csv name; GCS objects add the .gz suffix.
        input_path = f"{INPUT_DIR}/{FILENAME}.gz"

        # Read each compressed source object directly from Google Cloud Storage.
        current_df = pd.read_csv(
            input_path,
            sep=";",
            compression="gzip",
            storage_options=GCS_STORAGE_OPTIONS,
            dtype={
                "LATITUDE": "string",
                "LONGITUDE": "string",
            },
        )

        source_frames.append(current_df)

    # Some target areas combine multiple provincial source files.
    df = pd.concat(
        source_frames,
        ignore_index=True,
    )

    # Merged provinces share some grid cells,
    # so the same record is delivered by more than one source file.
    if len(source_frames) > 1:
        df = df.drop_duplicates(ignore_index=True)

    monthly = transform_province_dataframe(
        df=df,
        source_name=FILENAME,
    )

    monthly["PROVINCE"] = row[2]
    monthly["REGION"] = row[3]

    monthly_datasets.append(monthly)
    print(row[2], "DONE")

Agrigento DONE
Alessandria DONE
Ancona DONE
Arezzo DONE
Ascoli Piceno DONE
Asti DONE
Avellino DONE
Bari DONE
Barletta-Andria-Trani DONE
Belluno DONE
Benevento DONE
Bergamo DONE
Biella DONE
Bologna DONE
Brescia DONE
Brindisi DONE
Caltanissetta DONE
Campobasso DONE
Caserta DONE
Catania DONE
Catanzaro DONE
Chieti DONE
Como DONE
Cosenza DONE
Cremona DONE
Crotone DONE
Cuneo DONE
Enna DONE
Fermo DONE
Ferrara DONE
Firenze DONE
Foggia DONE
Forlì-Cesena DONE
Frosinone DONE
Genova DONE
Gorizia DONE
Grosseto DONE
Imperia DONE
Isernia DONE
La Spezia DONE
L'Aquila DONE
Latina DONE
Lecce DONE
Lecco DONE
Livorno DONE
Lodi DONE
Lucca DONE
Macerata DONE
Mantova DONE
Massa-Carrara DONE
Matera DONE
Messina DONE
Milano DONE
Modena DONE
Monza e della Brianza DONE
Napoli DONE
Novara DONE
Oristano DONE
Nuoro DONE
Padova DONE
Palermo DONE
Parma DONE
Pavia DONE
Perugia DONE
Pesaro e Urbino DONE
Pescara DONE
Piacenza DONE
Pisa DONE
Pistoia DONE
Pordenone DONE
Potenza DONE
Prato DONE
Provincia Autonoma di Bolzan

## Final assembly and local export

The transformed monthly DataFrames are concatenated into one complete climate dataset and saved in the local project `outputs/` directory.

This final export is local; the raw gzip-compressed objects stored in Google Cloud Storage are only read and are not modified.


In [6]:
# Combine the monthly results for all mapped production areas
final_dataset = pd.concat(
    monthly_datasets,
    ignore_index=True,
)

In [7]:
# Save the complete transformed climate dataset locally
output_path = OUTPUT_DIR / "climate_full_dataset.csv"

final_dataset.to_csv(
    output_path,
    index=False,
)